# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Seminar 5: Clasificare: arena modelelor</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/05-clasificare/seminar-05-clasificare.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "05-clasificare"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Problema de business

Operatorul de telecomunicații: *„Care clienți urmează să plece?”*

- **Rolul vostru:** analist de date: comparați mai multe modele de clasificare și explicați ce fel de greșeli face fiecare.
- **Ce predați la final:** comparația a cel puțin trei modele pe `lab-set1` și pe **setul proiectului vostru**, cu modelul de referință, matricea de confuzie a modelului ales și o recomandare argumentată.

Azi este seminarul cel mai apropiat de proiect: **arena** din `curs-05b`, rulată de voi, mai întâi pe `lab-set1`, apoi pe datele voastre.

## Din curs: ce aplicăm azi

**Concepte cheie** (modulul 5)

1. **Probabilitate și prag:** clasificatorul dă o probabilitate; pragul o transformă în clasă, iar 0,5 nu e o lege.
2. **Referința și împărțirea stratificată:** acuratețea se citește față de clasa majoritară.
3. **Matricea de confuzie:** fals pozitivi și fals negativi au costuri diferite; recall, precizie; AUC compară modele fără prag.
4. **Cum decid modelele:** regresia logistică (sigmoidă), kNN (vecini; cere scalare), arborele (reguli; memorează fără limită de adâncime), random forest și gradient boosting (mulți arbori), SVM (marginea cea mai largă; cere scalare).
5. **Arena:** același protocol (împărțire, `Pipeline`, referință, metrici) pentru toate modelele, într-un singur tabel; supraînvățarea se vede ca diferență mare între antrenare și test.

**Ce ceri asistentului AI** (exemplu de cerere)

> Pe `lab-set1`, fără `customerID`, cu `TotalCharges` transformat în număr (golurile = 0), împarte 80/20 stratificat (`random_state=42`). Într-un `Pipeline` cu completarea lipsurilor, one-hot și (unde trebuie) standardizare, compară clasa majoritară, regresia logistică, un arbore cu `max_depth=5` și random forest. Pentru fiecare: acuratețea pe antrenare și pe test, recall, precizie și AUC pe test, într-un singur tabel; apoi matricea de confuzie pentru modelul cel mai bun.

**Ce verifici înainte să accepți rezultatul**

- [ ] ID-urile sunt scoase, iar coloanele „numerice” stocate ca text au fost convertite;
- [ ] împărțirea este stratificată, iar pregătirea e în `Pipeline`, învățată doar pe antrenare;
- [ ] tabelul are un rând „clasa majoritară”;
- [ ] te uiți la recall, precizie și AUC, nu doar la acuratețe;
- [ ] compari antrenarea cu testul (un arbore nelimitat?);
- [ ] poți explica, pentru modelul ales, cum decide și ce greșeli face.

## Cum lucrăm azi (2 ore)

| Timp | Activitate |
|---|---|
| 15 min | **Demonstrație ghidată:** cererea de mai sus, rulată și verificată împreună |
| 45 min | **Exerciții** 1-3 pe `lab-set1`: voi scrieți cererile, rulați, verificați, interpretați |
| 50 min | **Arena pe proiectul vostru:** același protocol, cel puțin trei modele |
| 10 min | **Discuție:** ce a greșit asistentul și cum v-ați dat seama |

Pentru fiecare exercițiu: scrieți cererea **în cuvintele voastre** (vezi rețeta din [ghidul AI](https://github.com/sbildea/SEED/blob/main/docs/ghid-programare-asistata-ai.md)), lipiți codul generat în celula de cod, rulați, bifați verificările și răspundeți la întrebarea de interpretare. Lipiți asistentului și **primerul** de la începutul `curs-05a`: îi spune regulile modulului.

**Sfat:** cereți asistentului **o funcție `evalueaza(model, nume)`** ca în `curs-05b`, care antrenează un model și adaugă un rând într-un tabel. Apoi fiecare model nou este o singură linie.

In [ ]:
from seed.date import incarca, lista_seturi
lista_seturi()

## Demonstrație ghidată

Cererea din secțiunea de mai sus, rulată împreună. Lipiți aici codul generat de asistent și comparați tabelul cu clasamentul din `curs-05b` (capitolul 9).

In [ ]:
# codul generat pentru cererea din demonstrație

## Exercițiul 1: Pregătirea și referința

**Ce cereți:** Pe `lab-set1`: scoateți `customerID`, transformați `TotalCharges` în număr (ce faceți cu golurile și de ce?), faceți împărțirea 80/20 stratificată (`random_state=42`) și calculați acuratețea și recall-ul modelului de referință (clasa majoritară).

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 1

**Verificare:** am bifat lista de verificare de mai sus? Ce am corectat?

**Interpretare:** De ce nu e suficient să spunem „modelul are acuratețe 79%”? Față de ce număr o comparați?

*Răspunsul meu:*

## Exercițiul 2: Trei modele în arenă

**Ce cereți:** Cereți funcția `evalueaza` și adăugați în același tabel: referința, regresia logistică, kNN (cu standardizare) și un arbore de decizie cu `max_depth` ales de voi (de exemplu 3, 5 sau 8). Pentru fiecare: acuratețe pe antrenare și pe test, recall, precizie și AUC pe test.

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 2

**Verificare:** am bifat lista de verificare de mai sus? Ce am corectat?

**Interpretare:** Care model supraînvață (diferență mare între antrenare și test)? Care are cel mai mare AUC? Alegeți unul și explicați, în două propoziții, **cum decide**.

*Răspunsul meu:*

## Exercițiul 3: Pragul și matricea de confuzie

**Ce cereți:** Pentru modelul ales la exercițiul 2: matricea de confuzie la pragul 0,5 și un tabel cu numărul de clienți contactați, recall și precizie pentru pragurile 0,2, 0,3, 0,4 și 0,5.

**Cererea mea (prompt):**

> *scrieți aici cererea trimisă asistentului*

In [ ]:
# codul generat pentru exercițiul 3

**Verificare:** am bifat lista de verificare de mai sus? Ce am corectat?

**Interpretare:** Ce prag ați alege dacă a pierde un client costă de 5 ori mai mult decât o ofertă trimisă degeaba? Argumentați cu cifrele din tabel.

*Răspunsul meu:*

## Arena pe proiectul vostru

Acesta este **nucleul proiectului**. Pe setul vostru de date (`incarca("proiect-NN")`):

1. **Pregătirea:** ținta (câte clase, în ce proporții?), ID-urile scoase, coloanele numerice stocate ca text convertite, valorile lipsă tratate cu o explicație.
2. **Împărțirea:** 80/20, stratificată, cu `random_state` fixat.
3. **Arena:** referința plus **cel puțin trei modele** din curs, prin aceeași funcție `evalueaza`, într-un singur tabel (acuratețe pe antrenare și pe test, recall, precizie, AUC).
4. **Alegerea:** matricea de confuzie a modelului ales și 3 propoziții: de ce acesta, cum decide, ce greșeli face (pe cine ratează?).

Dacă setul vostru e foarte mare (sute de mii de rânduri), cereți asistentului să lucreze pe un eșantion stratificat (de exemplu 20.000 de rânduri) pentru kNN și SVM.

In [ ]:
# codul pentru proiectul vostru

## Jurnal AI (pe scurt)

| Exercițiu | Cererea principală | Ce am verificat / corectat |
|---|---|---|
| Demonstrație |  |  |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| Proiect |  |  |